In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error

pd.set_option('display.float_format', '{:,.0f}'.format)
plt.rcParams['figure.dpi'] = 100

print("Bibliotēkas ielādētas ✓")

In [ ]:
df = pd.read_csv('../week2/apartments_clean.csv')
print(f"Datu kopa: {df.shape[0]} dzīvokļi, {df.shape[1]} kolonnas")
print(df.head(3))

In [ ]:
cluster_features = ['price', 'area', 'rooms', 'floor']
X_cluster = df[cluster_features].copy()

print("Pazīmes klasterizācijai:")
print(X_cluster.describe().round(0))

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

print("Pēc normalizācijas:")
print(pd.DataFrame(X_scaled, columns=cluster_features).describe().round(2))

"""
## 1.4. Kāpēc šādas pazīmes un kāpēc normalizācija?

**Kāpēc neiekļāvām district un series?**
K-Means mēra eiklīda attālumu starp punktiem skaitliskās dimensijās.
Kategoriskām vērtībām kā "Centrs" vai "Hruščova projekts" nav matemātiskas
"tuvāk" un "tālāk" jēgas — tās ir nosaukumi, nevis skaitļi. One-Hot
enkodētās kategorijas radītu simtiem dimensiju un "izskalotu" klasterus.

**Kāpēc neiekļāvām price_per_meter?**
`price_per_meter` ir aprēķināts kā `price / area` — tā ir dublēta informācija.
Ja modelis redz gan `price`, gan `area`, gan `price_per_meter`, tad cenas
un platības ietekme tiek mākslīgi trīskāršota.

**Kāpēc normalizācija ir kritiski svarīga?**
`price` ir desmitiem tūkstošu eiro, bet `rooms` ir 1–5. Bez normalizācijas
cena dominētu visus klasteru lēmumus un istabu skaits būtu praktiski
neredzams. StandardScaler izlīdzina visus features uz vienādu mērogu.
"""

In [ ]:
results = {}

for K in [3, 5]:
    km = KMeans(n_clusters=K, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    results[K] = {
        'labels': labels,
        'inertia': km.inertia_,
        'sizes': pd.Series(labels).value_counts().sort_index().tolist()
    }
    print(f"K={K}: inertia = {km.inertia_:.0f}, "
          f"klasteru izmēri = {results[K]['sizes']}")

df['cluster_K3'] = results[3]['labels']
df['cluster_K5'] = results[5]['labels']

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, K in zip(axes, [3, 5]):
    for cluster_id in sorted(df[f'cluster_K{K}'].unique()):
        cluster_data = df[df[f'cluster_K{K}'] == cluster_id]
        ax.scatter(cluster_data['area'], cluster_data['price'] / 1000,
                   alpha=0.5, s=20,
                   label=f'Klasteris {cluster_id} (n={len(cluster_data)})')
    ax.set_xlabel('Platība (m²)')
    ax.set_ylabel('Cena (tūkstošos €)')
    ax.set_title(f'K = {K}')
    ax.legend(loc='upper left', fontsize=9)

plt.suptitle('Salīdzinājums: 3 vs. 5 segmenti', fontsize=14)
plt.tight_layout()
plt.show()

"""
## 2.4. K=3 vai K=5? Biznesa izvēle

**K=3 novērojums:** Trīs skaidri atšķirīgi segmenti — lētie, vidējie un dārgie
dzīvokļi. Vienkārši interpretējami, bet iespējams pārāk rupji — premium
Centra dzīvokļi tiek sajaukti ar vidusmēra segmentu.

**K=5 novērojums:** Piecu segmentu sadalījums atklāj papildu nianses —
luksusa Centra dzīvokļi tiek atsevišķi identificēti, un var redzēt atšķirību
starp mazajiem un lielākajiem guļamrajonu dzīvokļiem.

**Mana izvēle: K=5.**
Latio tirgus pētniekam ir vajadzīgi precīzi segmenti, ko var prezentēt
klientiem. Ar 3 segmentiem mēs zaudētu svarīgu informāciju par premium
segmentu. 5 segmenti dod pietiekami detalizētu, bet vēl saprotamu ainu —
mārketinga komanda var strādāt ar 5 skaidri nosauktām grupām.
"""

In [ ]:
K = 5
df['cluster'] = df[f'cluster_K{K}']

profile = df.groupby('cluster').agg(
    count=('price', 'size'),
    median_price=('price', 'median'),
    median_area=('area', 'median'),
    median_rooms=('rooms', 'median'),
    median_floor=('floor', 'median'),
    median_price_per_m2=('price_per_meter', 'median'),
).round(0)

print("=== KLASTERU PROFILI ===")
print(profile)

In [ ]:
print("\n=== TOP 3 RAJONI KATRĀ KLASTERĪ ===")
for c in sorted(df['cluster'].unique()):
    top = df[df['cluster'] == c]['district'].value_counts().head(3)
    print(f"\nKlasteris {c}:")
    print(top.to_string())

print("\n=== TOP 3 MĀJAS TIPI KATRĀ KLASTERĪ ===")
for c in sorted(df['cluster'].unique()):
    top = df[df['cluster'] == c]['series'].value_counts().head(3)
    print(f"\nKlasteris {c}:")
    print(top.to_string())

In [ ]:
heatmap_data = profile[['median_price', 'median_area', 'median_rooms',
                         'median_floor', 'median_price_per_m2']].T

heatmap_norm = heatmap_data.sub(heatmap_data.min(axis=1), axis=0).div(
    heatmap_data.max(axis=1) - heatmap_data.min(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(heatmap_norm, annot=heatmap_data, fmt=',.0f',
            cmap='RdYlGn_r',
            cbar_kws={'label': 'Relatīvā vērtība'}, ax=ax)
ax.set_title(f'Klasteru profili (K={K}) — vērtības anotētas, krāsas = relatīvi')
ax.set_xlabel('Klasteris')
plt.tight_layout()
plt.show()

"""
## 3.5. Klasteru nosaukumi un biznesa interpretācija

### Klasteris 0 — "Lētās hruščovkas guļamrajonos"
- **Tipisks dzīvoklis:** 2 ist., ~45 m², Hruščova projekts, 3. stāvs
- **Cena:** ~35 000–45 000 € (~800–900 €/m²)
- **Kur atrodas:** Ķengarags, Imanta, Purvciems
- **Tipiskais pircējs:** Pirmā mājokļa pircējs ar ierobežotu budžetu,
  investors īrei
- **Latio rekomendācija:** Pozicionēt kā "ekonomisko sākuma līmeni";
  izcelt zemo ieejas slieksni un ātro pārdošanas ātrumu

### Klasteris 1 — "Vidusmēra ģimenes dzīvokļi"
- **Tipisks dzīvoklis:** 3 ist., ~60 m², Lietuviešu/Padomju projekts,
  4. stāvs
- **Cena:** ~55 000–75 000 € (~950 €/m²)
- **Kur atrodas:** Imanta, Ziepniekkalns, Āgenskalns
- **Tipiskais pircējs:** Jauna ģimene, kas meklē lielāku platību par
  saprātīgu cenu
- **Latio rekomendācija:** Uzsvērt istabu skaitu un ģimenes piemērotību;
  piedāvāt salīdzinājumu ar īres izmaksām

### Klasteris 2 — "Renovētie vidējā cenu segmenta dzīvokļi"
- **Tipisks dzīvoklis:** 2–3 ist., ~55 m², renovēts, 3.–5. stāvs
- **Cena:** ~80 000–110 000 € (~1 500–1 800 €/m²)
- **Kur atrodas:** Āgenskalns, Teika, Mežciems
- **Tipiskais pircējs:** Profesionālis, kurš vēlas komfortu bez Centra
  cenas
- **Latio rekomendācija:** Izcelt renovācijas kvalitāti un enerģoefektivitāti;
  salīdzināt ar līdzīgiem Centra dzīvokļiem

### Klasteris 3 — "Premium Centra dzīvokļi"
- **Tipisks dzīvoklis:** 2–3 ist., ~65 m², Jaunceltne/Renovēts, 4. stāvs
- **Cena:** ~150 000–250 000 € (~2 500–3 500 €/m²)
- **Kur atrodas:** Centrs, Vecrīga
- **Tipiskais pircējs:** Augsta ienākuma profesionālis, ārzemju investors,
  tūristu īres investors
- **Latio rekomendācija:** Prezentēt kā investīciju ar augstu īres ienesīgumu;
  izcelt atrašanās vietu un prestiža faktoru

### Klasteris 4 — "Lielās platības dzīvokļi"
- **Tipisks dzīvoklis:** 4–5 ist., ~90 m², dažādi tipi, 3. stāvs
- **Cena:** ~90 000–130 000 € (~1 100 €/m²)
- **Kur atrodas:** Dažādi rajoni
- **Tipiskais pircējs:** Liela ģimene, kas vajag vietu, vai investors
  komunālajai īrei
- **Latio rekomendācija:** Izcelt kvadratūru/cenas attiecību; pozicionēt
  kā "vairāk par mazāku naudu nekā Centrā"
"""

In [ ]:
budget_70k = df[df['price'] <= 70000]
print("=== Jautājums 1: Dzīvokļi līdz 70 000 € ===")
print(f"Kopā sludinājumi: {len(budget_70k)}")
print("Pa klasteriem:")
print(budget_70k['cluster'].value_counts().sort_index())

# Jautājums 2: 3-istabu, zem 100 000 €, ne hruščovka
q2 = df[(df['rooms'] >= 3) &
        (df['price'] <= 100000) &
        (~df['series'].str.contains('Hruščova', na=False))]
print(f"\n=== Jautājums 2: 3+ ist., <100k€, ne hruščovka ===")
print(f"Atbilstošie sludinājumi: {len(q2)}")
print("Pa klasteriem:")
print(q2['cluster'].value_counts())

# Jautājums 3: Centrs, budžets 250 000 €
q3 = df[(df['district'] == 'Centrs') & (df['price'] <= 250000)]
print(f"\n=== Jautājums 3: Centrs, līdz 250 000 € ===")
print(f"Sludinājumu skaits: {len(q3)}")
print(f"Vai >50 sludinājumi? {'JĀ ✓' if len(q3) > 50 else 'NĒ ✗'}")

"""
## 4.1. Atbildes uz Latio klientu jautājumiem

**1. IT speciālists, budžets līdz 70 000 €:**
Ieteicamais segments ir **Klasteris 0** ("Lētās hruščovkas guļamrajonos")
un daļa no **Klastera 1** (vidusmēra ģimenes dzīvokļi). Šajā cenu diapazonā
ir pieejams plašs izvēles klāsts — galvenokārt Ķengarags, Imanta, Purvciems.
Mediānā cena šajos segmentos ir ~40 000–65 000 €, kas ietilpst budžetā.

**2. Ģimene ar 3 bērniem, 3-istabu, zem 100 000 €, ne hruščovka:**
Atbilstošais segments ir **Klasteris 1** un **Klasteris 2**.
Jautājums 2: Atbilstošo dzīvokļu skaits ir 662 — galvenokārt Klasterī 2 (514 dzīvokļi) un Klasterī 3 (142 dzīvokļi).
Šādu dzīvokļu ir pietiekami daudz — galvenokārt Imantā, Ziepniekkalnā
un Āgenskalnā.

**3. Investors ar 250 000 €, Centrs, tūristu īre:**
Ideāli piemērots **Klasteris 3** ("Premium Centra dzīvokļi").
Visvairāk pārcenotu īpašumu ir Klasterī 3 (25%) un Klasterī 0 un 2 (24%). Centrs ir heterogēnākais segments — pārbaudi vai >50.
Centrā šajā cenu līmenī ir laba izvēle, un īres ienesīgums tūristiem
ir augsts pateicoties atrašanās vietai.
"""

In [ ]:
df['price_vs_cluster_median'] = df['price'] / df.groupby('cluster')['price'].transform('median')

overpriced = df[df['price_vs_cluster_median'] > 1.5].groupby('cluster').size()
total = df.groupby('cluster').size()
overpriced_pct = (overpriced / total * 100).round(1).fillna(0)

print("Sludinājumu īpatsvars >1.5x klastera mediānas (potenciāli pārcenoti):")
print(overpriced_pct)

"""
## 4.2. Pārmaksas analīze — secinājumi

**Kurš klasteris satur visvairāk potenciāli pārcenotu īpašumu?**
[Aizpildi pēc šūnas 11 rezultātiem]

Visticamāk tas ir **Klasteris 3** (Premium Centrs) — šajā segmentā
cenu diapazons ir ļoti plašs, jo "Centrs" ietver gan vienkāršus
padomju laikmeta dzīvokļus, gan luksusa jaunceltnes. Klasterizācija
šos divus apakštirgus apvieno vienā klasterī, tāpēc augstās cenas
šeit var gan atspoguļot patiesu luksusa vērtību, gan arī pārmērīgi
augstu sludinājuma cenu salīdzinājumā ar līdzīgiem dzīvokļiem.

Latio rekomendācija: sludinājumus ar `price_vs_cluster_median > 1.5`
automātiski atzīmēt ar "Cena virs tirgus vidējā — pārbaudiet atbilstību".
"""

In [ ]:
features = ['rooms', 'area', 'floor', 'district', 'series']
X = pd.get_dummies(df[features], columns=['district', 'series'], drop_first=True)
y = df['price']

# Vienreizējs split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf = RandomForestRegressor(n_estimators=200, max_depth=15,
                           random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
single_mae = mean_absolute_error(y_test, rf.predict(X_test))
print(f"Vienreizējs split MAE: €{single_mae:,.0f}")

In [ ]:
rf_cv = RandomForestRegressor(n_estimators=200, max_depth=15,
                              random_state=42, n_jobs=-1)
scores = cross_val_score(rf_cv, X, y,
                         cv=5, scoring='neg_mean_absolute_error',
                         n_jobs=-1)
mae_scores = -scores

print("5-fold CV MAE rezultāti pa fold'iem:")
for i, mae in enumerate(mae_scores, 1):
    print(f"  Fold {i}: €{mae:,.0f}")
print(f"\nVidējā MAE: €{mae_scores.mean():,.0f}")
print(f"Std:        €{mae_scores.std():,.0f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
folds = [f'Fold {i}' for i in range(1, 6)]
ax.bar(folds, mae_scores / 1000, color='#2E86AB', alpha=0.7)
ax.axhline(mae_scores.mean() / 1000, color='red', linestyle='--',
           linewidth=2, label=f'CV vidējā: €{mae_scores.mean():,.0f}')
ax.axhline(single_mae / 1000, color='orange', linestyle='--',
           linewidth=2, label=f'Vienreizējs split: €{single_mae:,.0f}')
ax.set_ylabel('MAE (tūkstošos €)')
ax.set_title('Cik stabils ir mūsu cenu prognozēšanas modelis?')
ax.legend()
plt.tight_layout()
plt.show()

"""
## 5.5. Cross-validation — biznesa interpretācija

**Vienreizējais MAE vs. CV vidējais:**
Modelis ir stabils — starpība starp fold'iem ir maza 
Ja vienreizējais MAE ir zemāks par CV vidējo — mums "paveicās" ar labu
split. Ja augstāks — bija "neveiksmīgs" split.

**CV standartnovirze eiro:**
CV standartnovirze €921 — tas nozīmē, ka atkarībā no datu sadalījuma modelis var kļūdīties par šo summu vairāk vai mazāk. Ļoti maza novirze — modelis ir stabils.

**Ko teiktu City24.lv direktoram?**
Mūsu modelis stabili prognozē ar MAE ~€17,871 ± €921 € dažādos
datu sadalījumos. Tas NAV tikai viens laimīgs split — cross-validation
apliecina, ka modelis ir konsekventāks nekā viena testa kopas rezultāts
varētu liecināt. Tomēr mērķis MAE < 15 000 nav sasniegts..
"""

In [ ]:
scaler_bad = StandardScaler()
X_scaled_bad = scaler_bad.fit_transform(X)

X_train_bad, X_test_bad, y_train_bad, y_test_bad = train_test_split(
    X_scaled_bad, y, test_size=0.2, random_state=42
)

rf_bad = RandomForestRegressor(n_estimators=200, max_depth=15,
                               random_state=42, n_jobs=-1)
rf_bad.fit(X_train_bad, y_train_bad)
mae_bad = mean_absolute_error(y_test_bad, rf_bad.predict(X_test_bad))
print(f"Ar data leakage MAE: €{mae_bad:,.0f}  ← optimistiski izpušķots!")

In [ ]:
pipe = make_pipeline(
    StandardScaler(),
    RandomForestRegressor(n_estimators=200, max_depth=15,
                          random_state=42, n_jobs=-1)
)

scores_clean = cross_val_score(pipe, X, y, cv=5,
                               scoring='neg_mean_absolute_error',
                               n_jobs=-1)
mae_clean = -scores_clean
print(f"Ar Pipeline CV MAE: €{mae_clean.mean():,.0f} ± €{mae_clean.std():,.0f}")

In [ ]:
comparison = pd.DataFrame({
    'Metode': ['Ar data leakage', 'Pareizs Pipeline'],
    'MAE': [f"€{mae_bad:,.0f}", f"€{mae_clean.mean():,.0f}"],
    'Uzticamība': ['BĪSTAMI — optimistiski izpušķots', 'DROŠS — patiess']
})
print(comparison.to_string(index=False))

"""
## 6.4. Data leakage un Pipeline — skaidrojums

**Kāpēc mērogošana pirms split rada problēmu?**
`StandardScaler.fit()` aprēķina vidējo un standartnovirzi no VISIEM datiem,
ieskaitot testa kopas vērtības. Tādējādi skalers "zina" par testa datiem
pirms modelis tos "redz" — tas ir datu noplūde.

**Šajā gadījumā (RandomForest + StandardScaler):**
RandomForest ir koku modelis — tas nav jūtīgs pret datu mērogu, jo koki
neizmanto attālumus, bet gan sadalīšanas slieksņus. Tāpēc leakage šeit
praktiski neietekmē MAE. Taču, ja izmantotu `LinearRegression` vai `KNN`
(kas ir mērogam jutīgi), leakage radītu ievērojami optimistiskākus rezultātus.

**Kāpēc Pipeline ir labākais risinājums?**
Pipeline automātiski nodrošina pareizu secību katrā CV fold'ā — skalers
tiek fit tikai uz treniņa datiem. Turklāt Pipeline samazina koda sarežģītību
un novērš cilvēciskās kļūdas.
"""

In [ ]:
pipe_grid = make_pipeline(
    StandardScaler(),
    RandomForestRegressor(random_state=42, n_jobs=-1)
)

param_grid = {
    'randomforestregressor__n_estimators': [100, 200, 400],
    'randomforestregressor__max_depth': [10, 15, 20]
}

grid = GridSearchCV(pipe_grid, param_grid, cv=3,
                    scoring='neg_mean_absolute_error',
                    n_jobs=-1, verbose=1)
grid.fit(X, y)

print(f"\n=== REZULTĀTI ===")
print(f"Labākie parametri: {grid.best_params_}")
print(f"Labākā CV MAE: €{-grid.best_score_:,.0f}")


"""
## 7.3. GridSearchCV — interpretācija

**Labākie parametri:** max_depth=20, n_estimators=400

**Uzlabojums salīdzinājumā ar sākotnējo konfigurāciju:**
Sākotnējā konfigurācija bija max_depth=15, n_estimators=200 ar MAE ~€17,871. GridSearchCV atrada labākos parametrus, bet uzlabojums ir minimāls — biznesa kontekstā nenozīmīgs.
MAE uzlabojums:  ~€17,871 — nenozīmīgsbiznesa kontekstā.

**Vai uzlabojums ir nozīmīgs?**
Ja MAE samazinās par €500 — tas ir nenozīmīgi (pircējs to nejūt).
Ja par €2 000+ — tas var nozīmēt dažu papildu dzīvokļu korektu cenu
novērtējumu, kas biznesa kontekstā ir vērtīgi.

**Kāpēc nedarām Grid Search ar 100 kombinācijām?**
Katrā kombinācijā modelis tiek trenēts cv=3 reizes — 100 kombinācijas
× 3 = 300 treniņi. Ar lielu datu kopu un sarežģītu modeli tas var
ilgt stundas. Reālā produkcijā izmanto RandomizedSearchCV vai
Bayesian optimization, kas ir ievērojami ātrāks.
"""